In [ ]:
# Load matches (inspect and undertand raw data format)
import pandas as pd
import numpy as np 
from statsbombpy import sb

matches = sb.matches(competition_id=43, season_id=106)
match_id = int(matches.iloc[0]["match_id"])
events = sb.events(match_id=match_id)

print(events.shape)
print(events.columns.tolist())
events.head()

(3211, 91)
['bad_behaviour_card', 'ball_receipt_outcome', 'ball_recovery_recovery_failure', 'block_deflection', 'block_offensive', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_right_foot', 'counterpress', 'dribble_nutmeg', 'dribble_outcome', 'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage', 'foul_committed_card', 'foul_committed_type', 'foul_won_advantage', 'foul_won_defensive', 'goalkeeper_body_part', 'goalkeeper_end_location', 'goalkeeper_outcome', 'goalkeeper_position', 'goalkeeper_technique', 'goalkeeper_type', 'id', 'index', 'injury_stoppage_in_chain', 'interception_outcome', 'location', 'match_id', 'minute', 'off_camera', 'out', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id', 'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_deflected', 'pass_end_location', 'pass_goal_assist', 'pass_height', 'pass_length', 'pass_miscommunication', 'pass_outcome', 'pass_outswinging', 'pass

,bad_behaviour_card,ball_receipt_outcome,ball_recovery_recovery_failure,block_deflection,block_offensive,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,...,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 3412, 'lineup': [{'player': {'id...",Serbia,786,00:00:00.000,Starting XI,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 4231, 'lineup': [{'player': {'id...",Switzerland,773,00:00:00.000,Starting XI,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Switzerland,773,00:00:00.000,Half Start,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Serbia,786,00:00:00.000,Half Start,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Serbia,786,00:00:00.000,Half Start,NaN


In [ ]:
# Investigative exercise for data features
print(events.shape)
print(events.columns.tolist())
events.head()

events["type"].value_counts().head(10)
events[events["type"] == "Carry"].head()

# Learnt from this that CARRIES eixist and that for each carry output we also have CONTEXTUAL 
# feature data such as location of carry, end point of carry, team, player, timestamp and
#  under pressure. 

type
Pass              966
Ball Receipt*     907
Carry             667
Pressure          177
Ball Recovery      92
Duel               68
Block              40
Clearance          36
Foul Committed     35
Foul Won           34
Name: count, dtype: int64

In [ ]:
# Isolate carries into own data for future feature engineering
carries = events[events["type"] == "Carry"].copy()

carry_cols = [
    "id",
    "match_id",
    "team",
    "player",
    "possession",
    "period",
    "minute",
    "second",
    "timestamp",
    "location",
    "carry_end_location",
    "under_pressure"
]

carries = carries[carry_cols]

print(carries.shape)
carries.head()

# Learnt that location and carry_end_location stored as [x,y] lists. Each carry can be linked to 
# a player, team, possession and time. whilst some can happen under pressure adn some carries have
# the same start and end location.


(667, 12)


,id,match_id,team,player,possession,period,minute,second,timestamp,location,carry_end_location,under_pressure
1879,0e5c867f-8c40-4754-bc81-014c8cf88b8d,3857256,Switzerland,Remo Freuler,2,1,0,1,00:00:01.742,"[43.1, 43.7]","[39.7, 45.5]",NaN
1880,a54c6781-c843-40bb-9973-cdef963cb218,3857256,Switzerland,Silvan Widmer,2,1,0,5,00:00:05.137,"[42.3, 72.2]","[42.3, 72.2]",NaN
1881,5992d383-954d-44af-9fb1-4f14899d6407,3857256,Switzerland,Djibril Sow,2,1,0,13,00:00:13.522,"[91.1, 76.4]","[105.6, 74.4]",NaN
1882,90093f2c-b161-4a08-9767-14b17f2ad745,3857256,Switzerland,Djibril Sow,4,1,0,48,00:00:48.188,"[100.3, 4.6]","[113.6, 12.4]",True
1883,e9fad88c-6795-4e10-93f0-edd6db302a68,3857256,Switzerland,Breel-Donald Embolo,4,1,0,51,00:00:51.771,"[106.0, 17.0]","[106.0, 17.0]",True


In [ ]:
# First feature-engineering step ( split locations into seperate columns for future engineeering - 
# coordiantes can be theier own number, allowing models to work)

carries["start_x"] = carries["location"].str[0]
carries["start_y"] = carries["location"].str[1]
carries["end_x"] = carries["carry_end_location"].str[0]
carries["end_y"] = carries["carry_end_location"].str[1] 

carries[["location", "carry_end_location", "start_x", "start_y", "end_x", "end_y"]].head()
carries[["start_x", "start_y", "end_x", "end_y"]].describe()

# Learnt that avg end_x typically larger than avg start_x meaning more often than not the 
# ball was carried forward.

,location,carry_end_location,start_x,start_y,end_x,end_y
1879,"[43.1, 43.7]","[39.7, 45.5]",43.1,43.7,39.7,45.5
1880,"[42.3, 72.2]","[42.3, 72.2]",42.3,72.2,42.3,72.2
1881,"[91.1, 76.4]","[105.6, 74.4]",91.1,76.4,105.6,74.4
1882,"[100.3, 4.6]","[113.6, 12.4]",100.3,4.6,113.6,12.4
1883,"[106.0, 17.0]","[106.0, 17.0]",106.0,17.0,106.0,17.0


In [ ]:
# Carry feature calculations
goal_x = 120
goal_y = 40

carries["carry_length"] = np.sqrt(
    (carries["end_x"] - carries["start_x"]) ** 2 +
    (carries["end_y"] - carries["start_y"]) ** 2
)

carries["start_distance_to_goal"] = np.sqrt(
    (goal_x - carries["start_x"]) ** 2 +
    (goal_y - carries["start_y"]) ** 2
)

carries["end_distance_to_goal"] = np.sqrt(
    (goal_x - carries["end_x"]) ** 2 +
    (goal_y - carries["end_y"]) ** 2
)

carries["distance_gain"] = (
    carries["start_distance_to_goal"] - carries["end_distance_to_goal"]
)

carries[
    [
        "player",
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "carry_length",
        "start_distance_to_goal",
        "end_distance_to_goal",
        "distance_gain"
    ]
].head()

carries[["carry_length", "start_distance_to_goal", "end_distance_to_goal"]].describe()

carries.sort_values("distance_gain", ascending=False) [
    ["player", "team", "start_x", "start_y", "end_x", "end_y", "carry_length", "distance_gain"]
].head(10)

# Can see now that the TOP distance gain shows very aggressive ball progressing actions, 
# whilst large carry_length don't mean more DANGEROUS actions as distance_gain and general
#  distance_from_goal needs to be considered

,player,start_x,start_y,end_x,end_y,carry_length,start_distance_to_goal,end_distance_to_goal,distance_gain
1879,Remo Freuler,43.1,43.7,39.7,45.5,3.847077,76.988960,80.488136,-3.499176
1880,Silvan Widmer,42.3,72.2,42.3,72.2,0.000000,84.107847,84.107847,0.000000
1881,Djibril Sow,91.1,76.4,105.6,74.4,14.637281,46.477629,37.292358,9.185271
1882,Djibril Sow,100.3,4.6,113.6,12.4,15.418495,40.512344,28.332314,12.180030
1883,Breel-Donald Embolo,106.0,17.0,106.0,17.0,0.000000,26.925824,26.925824,0.000000
